#Before to start
Put this notebook file in the submitted Google Drive directory.

Edit **Section 1**, i.e., the first part of this notebook file implementing:
1. **Section 1.1**: the loading of your best model(s):
  * a. Create the network(s)
  * b. Load the best saved model(s) using a relative path (e.g., `"./model.pth"` or `"./model/best.pth"`)
2. **Section 1.2**: the predict function of your model(s):
  * a. Pre-process the input batch of data
  * b. Perform the forward using the network(s)
  * c. Post-process the output of the network(s)


DO NOT EDIT **Section 2** since it replicates the exact evaluation code we run on the private test set.

To test your model you have to:
* create a directory named `"eval"` containing the test images
* run all cells

#Section 1: YOUR CODE

##Section 1.1: IMPLEMENT HERE THE FUNCTION TO LOAD YOUR MODEL
For example, here we use a simple network.

**IMPORTANT: load the trained weights of your model here!**

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/2026_MLinf_gr16/Data/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import torch
NUM_CLASSES=8
def load_model():
  weights_path = '/content/drive/MyDrive/2026_MLinf_gr16/Best_models/efficientnet_optimized.pth'
  model = torch.load(weights_path, map_location=torch.device('cpu'), weights_only=False)
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = model.to(device)
  model.eval()
  return model

##Section 1.2: IMPLEMENT HERE YOUR PREDICT FUNCTION

Consider that the input is a batch of data with:
```
shape = (batch_size, rows, cols, channels=3)
dtype = uint8
```
For example, here we implement a simple pre and post processing and we call the model forward.

**IMPORTANT: implement your own pre- and post-processing pipeline here!**

Note that you can use torchvision transformations on a single image of the batch. [Read the documentation](https://docs.pytorch.org/vision/main/transforms.html) for more details.

In [3]:
import torch
import torchvision.transforms as transforms
val_transforms = transforms.Compose([
    transforms.Resize(256, antialias=True),
    transforms.CenterCrop(224),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229,0.224,0.225])
])

def predict(model, X):
  '''
  X is a uint8 numpy tensor of size (batch_size, rows, cols, channels=3)
  The output must be a uint8 numpy tensor of size (batch_size, 1)
  '''

  # THIS IS AN EXAMPLE! REPLACE THIS FUNCTION WITH YOUR CODE
  device = next(model.parameters()).device
  model.eval()
  with torch.no_grad():
    X_tensor = torch.from_numpy(X)
    X_tensor = X_tensor.permute(0,3,1,2).float()
    X_tensor /= 255.0
    X_tensor = val_transforms(X_tensor)
    X_tensor=X_tensor.to(device)
    logits=model(X_tensor)
    Y= logits.argmax(dim=-1).unsqueeze(1).cpu().numpy().astype(np.uint8)
  return Y

Run this cell to verify that the produced output is well formatted

In [4]:
import numpy as np
model = load_model()
X = np.random.randint(0, 255, size=(2,256,256,3), dtype=np.uint8) # this batch size is used only as an example
y = predict(model,X)
assert (y.shape == (X.shape[0],1) and y.dtype == np.uint8 \
      and (y >= 0).all() and (y < NUM_CLASSES).all()), "Verify your model loading or predict function."

In [5]:
cartella_eval = "./eval"
if not os.path.exists(cartella_eval):
    os.makedirs(cartella_eval)
    print(f"Cartella '{cartella_eval}' creata con successo!")
else:
    print(f"La cartella '{cartella_eval}' esiste già.")

if len(cartella_eval) == 0:
    print("La cartella './eval' è VUOTA!")
else:
    print("Puoi procedere a runnare tutto.")

Cartella './eval' creata con successo!
Puoi procedere a runnare tutto.


#Before to run the code
1. eventually change the current working directory using the `os.chdir` function (DO IT IN THE EMPTY CELL BELOW THIS ONE)
2. create a directory named `"./eval"` in the current working directory
3. verify that the `"./eval"` directory contains the image files for the test

Then, run all the cells.

#Section 2: Test code (DO NOT MODIFY THE CODE BELOW!)
This is exactly the code we run for the final test.

Just run the code to verify that it works. This is the exact code we run for the final evaluation on the private test set.

In [ ]:
#from google.colab import drive;drive.mount('/content/drive'); import os; os.chdir("/content/drive/My Drive/Didattica/ML/exam_2025_2026/project_work_Ing_Inf/evaluation")

In [ ]:
import os
from glob import glob

test_dir = "./eval/" # Do not modify this path, instead create a directory with this name in the same folder of this test.ipynb file
assert os.path.isdir(test_dir), "The evaluation directory does not exist. Create it, put some images in it and run again this cell."

samples = [sample for sample in glob(test_dir + '/**', recursive=True) if os.path.isfile(sample)]
assert len(samples) > 0, "The evaluation directory is empty. Put some images in it and run again this cell."
print(f"Found {len(samples)} samples in {test_dir}")

In [ ]:
import os
from PIL import Image
import numpy as np
from tqdm import tqdm

# Run YOUR LOAD_MODEL FUNCTION
model = load_model()

# Main loop
verbose = True

PREDICTIONs = np.zeros((len(samples), )) * np.nan
for i, img_path in tqdm(enumerate(samples), desc="Processing samples"):
  try:  # ATTENTION: any error occurring in this try-catch means that the corresponding PREDICTION is evaluated as an ERROR of the neural network
    # Open images
    rgb_image = Image.open(img_path)
    rgb_array = np.asarray(rgb_image)[None, ...]
    if verbose:
      print(f"\n  Loaded {img_path}. The input batch has shape {rgb_array.shape} and dtype {rgb_array.dtype}")

    # Run YOUR PREDICT FUNCTION
    predicted_labels_array = predict(model, rgb_array).squeeze()
    if verbose:
      print(f"  Predicted label {predicted_labels_array}")

    PREDICTIONs[i] = predicted_labels_array

  except FileNotFoundError:
    print(f"  Error: Could not find image file {img_path}")
  except Exception as e:
    print(f"  Error processing image {img_path}: {e}")

PREDICTIONs = PREDICTIONs.astype(np.uint8)
print(f"Predictions: {PREDICTIONs}")
np.save("predictions.npy", PREDICTIONs)